In [1]:
import torch
import matplotlib.pyplot as plt
import datasets

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
dataset = datasets.load_dataset("google/code_x_glue_cc_code_refinement", "medium").shuffle()

In [4]:
def letter_tokenizer(s: str):
    return list(s) + ["<EOS>"]

def word_tokenizer(s: str):
    return s.split() + ["<EOS>"]

vocab = set()
def map_fn(x: dict[str, list[str]]):
    # x: {"buggy": [str], "fixed": [str]}
    vocab.update("".join(x["buggy"]) + "".join(x["fixed"]))
    return {
        "buggy": [letter_tokenizer(f) for f in x["buggy"]],
        "fixed": [letter_tokenizer(f) for f in x["fixed"]],
    }

dataset = dataset.map(map_fn, batched=True)
vocab.add("<EOS>")
vocab.add("<PAD>")  # Add padding token to the vocabulary
vocab.add("<SOS>")  # Add start-of-sequence token to the vocabulary

print(dataset["train"][0])

Map:   0%|          | 0/52364 [00:00<?, ? examples/s]

Map:   0%|          | 0/6546 [00:00<?, ? examples/s]

Map:   0%|          | 0/6545 [00:00<?, ? examples/s]

{'id': 49497, 'buggy': ['p', 'u', 'b', 'l', 'i', 'c', ' ', 'v', 'o', 'i', 'd', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '1', ' ', '(', ' ', ')', ' ', '{', ' ', 'j', 'a', 'v', 'a', '.', 'u', 't', 'i', 'l', '.', 'C', 'o', 'l', 'l', 'e', 'c', 't', 'i', 'o', 'n', 's', '.', 's', 'o', 'r', 't', ' ', '(', ' ', 'V', 'A', 'R', '_', '1', ' ', ',', ' ', 'n', 'e', 'w', ' ', 'T', 'Y', 'P', 'E', '_', '1', ' ', '<', ' ', 'T', 'Y', 'P', 'E', '_', '2', ' ', '>', ' ', '(', ' ', ')', ' ', '{', ' ', 'p', 'u', 'b', 'l', 'i', 'c', ' ', 'i', 'n', 't', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '2', ' ', '(', ' ', 'T', 'Y', 'P', 'E', '_', '2', ' ', 'V', 'A', 'R', '_', '2', ' ', ',', ' ', 'T', 'Y', 'P', 'E', '_', '2', ' ', 'V', 'A', 'R', '_', '3', ' ', ')', ' ', '{', ' ', 'r', 'e', 't', 'u', 'r', 'n', ' ', 'V', 'A', 'R', '_', '2', ' ', '.', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '3', ' ', '(', ' ', ')', ' ', '.', ' ', 'c', 'o', 'm', 'p', 'a', 'r', 'e', 'T', 'o', ' ', '(', ' ', 'V', 'A', 'R', '_', '3', ' ', '.', '

In [5]:
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Vocab: {vocab}")

Vocab size: 89
Vocab: {'a', 'f', 'q', '~', '0', 'N', '>', 'i', 'S', '.', 'G', '<EOS>', '%', 'p', '\\', 'O', 'u', 'I', '=', ',', 'V', ';', '1', '2', ' ', '{', '<', 'k', '*', '}', '7', '_', 'x', '"', 'A', 'H', '8', 'j', 'l', 'B', 'T', '<SOS>', '^', 'y', ':', 'M', 'E', 'D', 'm', 'w', '6', 'n', '\n', '-', '!', '|', '5', '<PAD>', 'F', 'd', ')', 'R', '+', 'v', 'U', '4', 'b', 'c', 'Y', 'h', 'C', ']', '?', '9', 't', 'W', '3', 's', 'g', '(', '[', 'r', 'o', 'z', 'e', 'L', 'P', '/', '&'}


In [6]:
OHE = {c: i for i, c in enumerate(vocab)}
def ohe_map_fn(x: dict[str, list[str]]):
    return {
        "buggy": [[OHE[c] for c in f]  for f in x["buggy"]],
        "fixed": [[OHE[c] for c in f]  for f in x["fixed"]],
    }

dataset = dataset.map(ohe_map_fn, batched=True, batch_size=5000, remove_columns=["id"])

Map:   0%|          | 0/52364 [00:00<?, ? examples/s]

Map:   0%|          | 0/6546 [00:00<?, ? examples/s]

Map:   0%|          | 0/6545 [00:00<?, ? examples/s]

In [7]:
print(dataset["train"][0])

{'buggy': [13, 16, 66, 38, 7, 67, 24, 63, 82, 7, 59, 24, 45, 46, 40, 35, 15, 47, 31, 22, 24, 79, 24, 60, 24, 25, 24, 37, 0, 63, 0, 9, 16, 74, 7, 38, 9, 70, 82, 38, 38, 84, 67, 74, 7, 82, 51, 77, 9, 77, 82, 81, 74, 24, 79, 24, 20, 34, 61, 31, 22, 24, 19, 24, 51, 84, 49, 24, 40, 68, 86, 46, 31, 22, 24, 26, 24, 40, 68, 86, 46, 31, 23, 24, 6, 24, 79, 24, 60, 24, 25, 24, 13, 16, 66, 38, 7, 67, 24, 7, 51, 74, 24, 45, 46, 40, 35, 15, 47, 31, 23, 24, 79, 24, 40, 68, 86, 46, 31, 23, 24, 20, 34, 61, 31, 23, 24, 19, 24, 40, 68, 86, 46, 31, 23, 24, 20, 34, 61, 31, 76, 24, 60, 24, 25, 24, 81, 84, 74, 16, 81, 51, 24, 20, 34, 61, 31, 23, 24, 9, 24, 45, 46, 40, 35, 15, 47, 31, 76, 24, 79, 24, 60, 24, 9, 24, 67, 82, 48, 13, 0, 81, 84, 40, 82, 24, 79, 24, 20, 34, 61, 31, 76, 24, 9, 24, 45, 46, 40, 35, 15, 47, 31, 76, 24, 79, 24, 60, 24, 60, 24, 21, 24, 29, 24, 29, 24, 60, 24, 21, 24, 29, 52, 11], 'fixed': [13, 16, 66, 38, 7, 67, 24, 63, 82, 7, 59, 24, 45, 46, 40, 35, 15, 47, 31, 22, 24, 79, 24, 37, 0, 6

In [8]:
def custom_collate_fn(batch):
    buggy = [torch.tensor(item["buggy"], dtype=torch.long) for item in batch]
    fixed = [torch.tensor(item["fixed"], dtype=torch.long) for item in batch]
    return torch.nn.utils.rnn.pad_sequence(buggy, batch_first=True, padding_value=OHE["<PAD>"]), torch.nn.utils.rnn.pad_sequence(fixed, batch_first=True, padding_value=OHE["<PAD>"])

train_dataloader, val_dataloader, test_dataloader = [
    torch.utils.data.DataLoader(
        dataset[split], batch_size=32, shuffle=True, collate_fn=custom_collate_fn
    ) for split in ["train", "validation", "test"]
]

print(f"Train: {len(train_dataloader.dataset)}")
print(next(iter(train_dataloader)))

Train: 52364
(tensor([[13, 81,  7,  ..., 57, 57, 57],
        [13, 81, 82,  ..., 57, 57, 57],
        [13, 16, 66,  ..., 57, 57, 57],
        ...,
        [13, 16, 66,  ..., 57, 57, 57],
        [13, 81,  7,  ..., 57, 57, 57],
        [13, 16, 66,  ..., 57, 57, 57]]), tensor([[13, 81,  7,  ..., 57, 57, 57],
        [13, 81, 82,  ..., 57, 57, 57],
        [13, 16, 66,  ..., 57, 57, 57],
        ...,
        [13, 16, 66,  ..., 57, 57, 57],
        [13, 81,  7,  ..., 57, 57, 57],
        [13, 16, 66,  ..., 57, 57, 57]]))


In [26]:
class LSTMModel(torch.nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=OHE["<PAD>"])
        self.encoder_lstm = torch.nn.LSTM(embedding_dim, hidden_dim, batch_first=True, num_layers=3)
        self.decoder_lstm = torch.nn.LSTM(embedding_dim, hidden_dim, batch_first=True, num_layers=3)
        self.classifier_mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, vocab_size)  # We don't want to predict the <PAD> token, so we reduce the output size by 1
        )

    def forward(self, x: torch.Tensor, y: torch.Tensor):
        inp_embedded = self.embedding(x)
        # We need to shift y to the right by one position and prepend a <SOS> token at the beginning for the decoder input
        sos_token = torch.full((y.size(0), 1), OHE["<SOS>"], dtype=torch.long, device=y.device)
        _, (h_n, c_n) = self.encoder_lstm(inp_embedded)
        # _ has output features (h_t) from the last layer of the LSTM for each timestep
        # h_n has the hidden state for the last timestep for each layer
        # c_n has the cell state for the last timestep for each layer
        # h_n, c_n are the context vectors from the all the lstm layers

        y_inp = torch.cat((sos_token, y), dim=1)
        y_embedded = self.embedding(y_inp)
        output, _ = self.decoder_lstm(y_embedded, (h_n, c_n))
        # We need output features (h_t) from the last layer of the LSTM for each timestep
        mlp_output = self.classifier_mlp(output[:, :-1])
        # TODO: try using different embeddings for encoder and decoder
        return mlp_output

    def evaluate(self, x: torch.Tensor):
        # Shape of x: (seq_len)
        self.eval()
        with torch.no_grad():
            inp_embedded = self.embedding(x)
            _, (h_n, c_n) = self.encoder_lstm(inp_embedded)
            outputs = [OHE["<SOS>"]]
            while True: # Keep feeding the output of the decoder back into itself until we generate an <EOS> token for all sequences in the batch
                y_embedded = self.embedding(torch.tensor([outputs[-1]], device=x.device))
                output, (h_n, c_n) = self.decoder_lstm(y_embedded, (h_n, c_n))  
                mlp_output = self.classifier_mlp(output[-1])
                next_token = mlp_output.argmax(dim=-1).item()
                outputs.append(next_token)
                if next_token == OHE["<EOS>"]:
                    break
            return outputs[1:]  # Exclude the initial <SOS> token

model = LSTMModel(vocab_size, embedding_dim=256, hidden_dim=512).to(DEVICE)

In [10]:
def train(model, train_dataloader, valid_dataloader, epochs=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=OHE["<PAD>"])  # Ignore the <PAD> token in the loss calculation

    losses = []
    valid_losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_dataloader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(X_batch, y_batch)
            loss = criterion(outputs.permute(0, 2, 1), y_batch)  # crossentropy expects (N, C, d1, d2, ...) for input and (N, d1, d2, ...) for target
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        losses.append(total_loss / len(train_dataloader))
        print(f"Epoch {epoch+1}/{epochs}, Loss: {losses[-1]:.4f}")

        model.eval()
        with torch.no_grad():
            valid_loss = 0
            for X_batch, y_batch in valid_dataloader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                outputs = model(X_batch, y_batch)
                loss = criterion(outputs.permute(0, 2, 1), y_batch)
                valid_loss += loss.item()
        valid_losses.append(valid_loss / len(valid_dataloader))
        print(f"Validation Loss: {valid_losses[-1]:.4f}")

    return losses, valid_losses

In [27]:
model.load_state_dict(torch.load('models/lstm.pth'))
# train_losses, valid_losses = train(model, train_dataloader, val_dataloader, epochs=12)
# torch.save(model.state_dict(), 'models/lstm.pth')
# plt.plot(train_losses, label='Train Loss')
# plt.plot(valid_losses, label='Validation Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.show()

<All keys matched successfully>

In [28]:
# Test loss
model.eval()
criterion = torch.nn.CrossEntropyLoss(ignore_index=OHE["<PAD>"])
test_loss = 0
with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        outputs = model(X_batch, y_batch)
        loss = criterion(outputs.permute(0, 2, 1), y_batch)
        test_loss += loss.item()
test_loss /= len(test_dataloader)
print(f"Test Loss: {test_loss:.4f}")

Test Loss: 0.2086


In [39]:
# try model on a sample from the test set

un_OHE = {i: c for c, i in OHE.items()}

sample = list(test_dataloader.dataset[0].values())

model.eval()
with torch.no_grad():
    outputs = model.evaluate(torch.tensor(sample[0], device=DEVICE))
print(f'BUGGY:                         {"".join([un_OHE[i] for i in sample[0] if i != OHE["<PAD>"]])}')  # Buggy code
print(f'Fixed:                         {"".join([un_OHE[i] for i in sample[1] if i != OHE["<PAD>"]])}')  # Fixed code
print(f'Model\'s predicted fixed code: {"".join([un_OHE[i] for i in outputs if i != OHE["<PAD>"]])}')  # Model's predicted fixed code

BUGGY:                         public double METHOD_1 ( java.lang.String VAR_1 ) throws java.io.IOException { TYPE_1 VAR_2 = TYPE_2 . get ( VAR_1 ) ; java.lang.String name = VAR_2 . getName ( ) ; if ( ( name . compareTo ( STRING_1 ) ) == 0 ) return - 1 ; TYPE_3 VAR_3 = VAR_2 . METHOD_2 ( ) . METHOD_3 ( ) ; java.lang.Double VAR_4 = VAR_3 . METHOD_4 ( ) ; return VAR_4 ; }
<EOS>
Fixed:                         public double METHOD_1 ( java.lang.String VAR_1 ) throws java.io.IOException { TYPE_1 VAR_2 = TYPE_2 . get ( VAR_1 ) ; if ( VAR_2 == null ) return - 1 ; java.lang.String name = VAR_2 . getName ( ) ; if ( ( name . compareTo ( STRING_1 ) ) == 0 ) return - 1 ; TYPE_3 VAR_3 = VAR_2 . METHOD_2 ( ) . METHOD_3 ( ) ; java.lang.Double VAR_4 = VAR_3 . METHOD_4 ( ) ; return VAR_4 ; }
<EOS>
Model's predicted fixed code: public double METHOD_1 ( java.lang.String VAR_1 ) throws TYPE_1 { java.lang.String VAR_2 = STRING_1 ; java.lang.String VAR_3 = STRING_2 ; java.lang.String VAR_4 = STRING_3 ; java

lowk impressive